# HOT-NERD Mouse Ileum Tutorial

This notebook demonstrates the complete HOT-NERD pipeline on a MERFISH mouse ileum dataset, performing cell segmentation refinement from spatial transcriptomics data.

## Dataset

**MERFISH Mouse Ileum Dataset**

This dataset uses MERFISH (Multiplexed Error-Robust Fluorescence In Situ Hybridization) spatial transcriptomics data from mouse ileum tissue. The initial cell segmentation was performed using **Baysor** with membrane prior derived from **Cellpose**.

**Reference:**
> Petukhov, V., Xu, R.J., Soldatov, R.A. et al. Cell segmentation in imaging-based spatial transcriptomics. *Nat Biotechnol* **40**, 345–354 (2022). https://doi.org/10.1038/s41587-021-01044-w

HOT-NERD refines the Baysor segmentation by leveraging NPMI-based quality metrics and graph-based transcript stitching to improve cell boundary accuracy and resolve ambiguous assignments.

## Pipeline Overview

The notebook executes the full HOT-NERD workflow:

1. **Conservative NPMI Pruning** - Remove low-quality transcripts based on gene co-expression patterns
2. **Component Annotation** - Graph-based detection and annotation of unassigned transcripts
3. **Hierarchical Stitching** - Merge partial cells and components into complete cell boundaries
4. **Spatial Coherence Enforcement** - Ensure cell assignments are spatially connected
5. **Final Refinement** - Re-stitch to produce final high-quality cell segmentation

## Performance Notes

This notebook uses **Cython-accelerated functions** (`_fast` variants) that reduce runtime from hours to minutes:
- `prune_transcripts_fast` with parallel processing
- `annotate_unassigned_components_fast` with bulk Cython pruning
- `enforce_spatial_coherence_fast` with C-level connected components

Run the next code cell to set up the Cython environment, imports, and helper functions.

In [14]:
# optional: install/enable Cython at runtime (run once if wheel not built)
%pip install --upgrade Cython

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pyximport, numpy, sys
import importlib
pyximport.install(setup_args={"include_dirs": [numpy.get_include()]},
                  language_level=3, build_in_temp=False)
sys.path.insert(0, "src")
import hotnerd.core as core
importlib.reload(core)   # ensures pyximport compilation runs
print("cython available:", core._cy_prune is not None)
print("cython _cy_spatial available:", getattr(core, "_cy_spatial", None) is not None)

/Users/lyuan13/anaconda3/envs/spatial/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cython available: True
cython _cy_spatial available: True


In [2]:
import sys
from pathlib import Path
import pandas as pd
import traceback

# ensure repo src on path
repo_root = Path.cwd()
sys.path.insert(0, str(repo_root / "src"))

# robust parquet reader helper
def read_parquet_robust(path: Path):
    try:
        return pd.read_parquet(path)
    except Exception as e:
        print("pd.read_parquet failed:", e)
        try:
            import pyarrow.parquet as pq
            import pyarrow as pa
            files = [path] if not path.is_dir() else sorted(path.glob("*.parquet"))
            tables = []
            for f in files:
                print("Reading parquet file:", f)
                pf = pq.ParquetFile(str(f))
                tbl = pf.read()
                tables.append(tbl)
            if not tables:
                raise RuntimeError("No parquet files found for fallback read")
            return pa.concat_tables(tables).to_pandas()
        except Exception:
            traceback.print_exc()
            raise


In [3]:
# Data paths (adjust if notebook root differs)
DATA_DIR = repo_root / "data"
PARQUET_PATH = DATA_DIR / "mouse_gut_df.parquet"
NPMI_CSV = DATA_DIR / "mouse_gut_npmi.csv"

print('parquet:', PARQUET_PATH)
print('npmi:', NPMI_CSV)

# Read data
print('Reading transcripts...')
df_transcripts = read_parquet_robust(PARQUET_PATH)
print('Transcripts rows:', len(df_transcripts))

print('Reading NPMI...')
df_npmi = pd.read_csv(NPMI_CSV)
print('NPMI rows:', len(df_npmi))

parquet: /Users/lyuan13/Desktop/HOT-NERD/tutorials/mouse_ileum/data/mouse_gut_df.parquet
npmi: /Users/lyuan13/Desktop/HOT-NERD/tutorials/mouse_ileum/data/mouse_gut_npmi.csv
Reading transcripts...
Transcripts rows: 819665
Reading NPMI...
NPMI rows: 47089


In [4]:
# Run conservative pruning
print('Running prune_transcripts (conservative NPMI pruning)')

# df_transcripts and df_npmi are loaded earlier in this notebook
df_pruned, aux = core.prune_transcripts_fast(
    df_transcripts,
    df_npmi,
    cell_id_col="cell_id",
    gene_col="feature_name",
    threshold=-0.1,
    unassigned_id="-1",
    n_jobs=-1,            # adjust for your machine (-1 for all cores handled inside)
    show_progress=True,  # enable tqdm progress bars
)
print('Done. pruned rows:', len(df_pruned))
df_pruned.head()

Running prune_transcripts (conservative NPMI pruning)


prune_pass2: 100%|██████████| 7103/7103 [00:00<00:00, 467856.11it/s]


Done. pruned rows: 819665


,transcript_id,feature_name,cell_id,x,y,z,is_noise,nuclei_probs,qv,overlaps_nucleus,cell_id_npmi_cons_p1,npmi_cons_p1_status,cell_id_npmi_cons_p2,npmi_cons_p2_status
0,3048145,Maoa,84,1705.0,1271.0,0.0,False,1.0,0.954363,1,84-1,partial_p1,84-1,partial_p2
1,3048147,Maoa,231,1725.0,1922.0,0.0,False,1.0,0.908246,1,231-1,partial_p1,-1,unassigned_from_partial
2,3048148,Maoa,231,1753.0,1863.0,0.0,False,1.0,0.977219,1,231-1,partial_p1,-1,unassigned_from_partial
3,3048149,Maoa,231,1760.0,1865.0,0.0,False,1.0,0.991316,1,231-1,partial_p1,-1,unassigned_from_partial
4,3048153,Maoa,53,1904.0,794.0,0.0,False,1.0,0.983210,1,53-1,partial_p1,53-1,partial_p2


In [5]:
df_final = core.annotate_unassigned_components_fast(
    df_pruned,            # DataFrame returned from prune_transcripts_fast
    aux,                  # aux dict returned from prune_transcripts_fast
    build_graph_fn=core.build_graph,
    prune_fn=core.prune_genes_by_npmi_greedy,
    coord_cols=("x","y","z"),
    k=8,
    dist_threshold=1.5,
    min_comp_size=50,
    npmi_threshold=-0.1,
    unassigned_final_col="cell_id_npmi_cons_p2",
    cell_id_col="cell_id",
    gene_col="feature_name",
    transcript_id_col="transcript_id",
    show_progress=True,   # enable tqdm
)

Constructed 44 edges among 91,352 transcripts (k≤8, d≤1.5 µm)


post_cc_mapping: : 4it [00:00, 707.81it/s]                        
grouping: 100%|██████████| 2/2 [00:00<00:00, 1851.79it/s]


In [6]:
df_stitched, entity_to_stitched = core.apply_stitching_to_transcripts_fast(
    df_final=df_final,          # DataFrame (output of annotate_unassigned_components/_fast)
    aux=aux,                    # aux dict from prune_transcripts(_fast)
    entity_col="cell_id_final",
    gene_col="feature_name",
    coord_cols=("x","y","z"),
    purity_threshold=0.05,
    penalize_simplicity=True,
    deltaC_min=0.0,
    use_3d=True,
    out_col="cell_id_stitched",
    show_progress=True,         # tqdm progress
)

stitching: 100%|██████████| 2/2 [00:19<00:00,  9.54s/it]


In [7]:
df_spatial = core.enforce_spatial_coherence_fast(
    df_stitched,                        # DataFrame produced by apply_stitching_to_transcripts(_fast)
    build_graph_fn=core.build_graph,
    entity_col="cell_id_stitched",      # label column to check
    coord_cols=("x", "y", "z"),
    k=5,
    dist_threshold=10.0,
    out_col="cell_id_spatial",
    show_progress=True,                 # show tqdm progress
)

Constructed 1,268,677 edges among 819,665 transcripts (k≤5, d≤10.0 µm)


spatial_labels: 100%|██████████| 10573/10573 [00:03<00:00, 2901.24it/s]


In [8]:
df_finetuned, entity_to_stitched_ft = core.apply_stitching_to_transcripts_fast(
    df_final=df_spatial,          # DataFrame (output of enforce_spatial_coherence_fast)
    aux=aux,
    entity_col="cell_id_spatial",
    gene_col="feature_name",
    coord_cols=("x", "y", "z"),
    purity_threshold=0.05,
    penalize_simplicity=True,
    deltaC_min=0.0,
    use_3d=True,
    out_col="cell_id_finetuned",
    show_progress=True,   
)

stitching: 100%|██████████| 2/2 [32:20<00:00, 970.49s/it] 


In [9]:
from pathlib import Path

out_dir = repo_root / "output"
out_dir.mkdir(parents=True, exist_ok=True)

fp_stitched = out_dir / "df_stitched.parquet"
fp_finetuned = out_dir / "df_finetuned.parquet"

df_stitched.to_parquet(fp_stitched, index=False)
df_finetuned.to_parquet(fp_finetuned, index=False)

print("Saved df_stitched ->", fp_stitched)
print("Saved df_finetuned ->", fp_finetuned)

Saved df_stitched -> /Users/lyuan13/Desktop/HOT-NERD/tutorials/mouse_ileum/output/df_stitched.parquet
Saved df_finetuned -> /Users/lyuan13/Desktop/HOT-NERD/tutorials/mouse_ileum/output/df_finetuned.parquet
